# Task 3 - Multi-Agent Financial Research System

Implements Task 3A (single tool-using agent), 3B (analyst/writer + critique loop), and 3C (memory + `agent_trace.jsonl`).


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'task3_agentic').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from task3_agentic.agents.research_agents import run_single_research_agent, run_multi_agent_pipeline, answer_followup
from task3_agentic.memory.store import ShortTermMemory, cache_path
from task3_agentic.observability import LOG_PATH

# Fresh run
p = cache_path('AAPL')
if p.exists():
    p.unlink()
single = run_single_research_agent('AAPL')
print('from_cache', single['from_cache'])
print('trace events', len(single['trace']))
print('report sections', list(single['report'].keys()))
single['trace'][:4]

from_cache False
trace events 11
report sections ['ticker', 'financial_health_summary', 'top_three_risks', 'hedge_strategy', 'sources_note']
[
  {
    "cycle": 0,
    "decision": "call_tool",
    "tool": "get_price_data",
    "args": {
      "ticker": "AAPL",
      "period": "2y"
    }
  },
  {
    "cycle": 0,
    "observation": {
      "ticker": "AAPL",
      "period": "2y",
      "rows": 501,
      "latest_close": 332.2699890136719,
      "sma_50": 317.97419982910156,
      "sma_200": 284.9533494567871,
      "rsi_14": 62.789009402560566,
      "macd": 3.226967132801235,
      "macd_hist": 1.3247165078741852,
      "bb_upper": 331.4729512408738,
      "bb_lower": 301.77904460873555
    }
  },
  {
    "cycle": 1,
    "decision": "call_tool",
    "tool": "calculate_volatility",
    "args": {
      "ticker": "AAPL",
      "window": 21
    }
  },
  {
    "cycle": 1,
    "observation": {
      "ticker": "AAPL",
      "window": 21,
      "daily_std": 0.014634322304162145,
      "annualised

## Observe/replan evidence and final report

In [1]:
print(single['report']['financial_health_summary'][:400])
print('Risks:', [r['title'] for r in single['report']['top_three_risks']])
print('Hedge:', single['report']['hedge_strategy'][:240])

AAPL latest close=332.2699890136719, SMA50=317.97419982910156, SMA200=284.9533494567871, RSI=62.789009402560566, ann. vol=0.23231266453667265, news sentiment=positive (0.3707865168539326).
Risks: ['Volatility regime risk', 'Trend / momentum reversal risk', 'Narrative / headline risk']
Hedge: Data-driven hedge: if annualised volatility is elevated versus the name's recent median and RSI is stretched, a partial protective put (or collar) sized to ~50% of notional for a 90-day horizon reduces left-tail exposure while leaving upsid


## Task 3B multi-agent + critique loop

In [1]:
multi = run_multi_agent_pipeline('AAPL')
for m in multi['messages']:
    print(m['role'], '->', m['content'])
print('Final risks:', len(multi['final_report']['top_three_risks']))

analyst -> Structured data brief ready
writer -> Clarification request
analyst -> Clarification response
writer -> Final research report
Final risks: 3


## Task 3C memory + observability

In [1]:
mem = ShortTermMemory()
for k,v in single['memory'].items():
    mem.set(k,v)
print(answer_followup('AAPL', 'What was the volatility reading?', mem))
second = run_single_research_agent('AAPL')
print('second run from_cache:', second['from_cache'])
print('agent_trace.jsonl lines:', len(Path(LOG_PATH).read_text().splitlines()))
print(Path(LOG_PATH).read_text().splitlines()[0][:300])

From session memory (no re-fetch): {'ticker': 'AAPL', 'window': 21, 'daily_std': 0.014634322304162145, 'annualised_volatility': 0.23231266453667265}
second run from_cache: True
agent_trace.jsonl lines: 10
{"timestamp": "2026-09-13T03:52:50.224165+00:00", "tool": "get_price_data", "inputs": {"ticker": "AAPL", "period": "2y"}, "output": "{\"ticker\": \"AAPL\", \"period\": \"2y\", \"rows\": 501, \"latest_close\": 332.2699890136719, \"sma_50\": 317.97419982910156, \"sma_200\": 284.9533494567871, \"rsi_14
